# Hill-Climbing Blend – Kaggle Playground S6E9

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

In [ ]:
INPUT = Path('input')
DATA_DIR = INPUT / 'competitions/playground-series-s6e9'
OOF_DIR = INPUT / 'oof'

TARGET = 'Will_Buy_EV'

## Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv', usecols=['id', TARGET])
y = (train[TARGET] == 'Yes').astype(int).to_numpy()
train_ids = train['id'].to_numpy()

test_ids = pd.read_csv(DATA_DIR / 'sample_submission.csv', usecols=['id'])['id'].to_numpy()

## Ensemble Members

In [ ]:
MEMBERS = {
    'lightgbm (ours)': ('base/oof_lgbm_base_margin.npy', None, 'base/test_lgbm_base_margin.npy', None),  # Unser Modell
    'heuljax-gen-ridge-lr': ('heuljax-gen-ridge-lr/oof_gen_ridge_lr.parquet', 'oof_pred', 'heuljax-gen-ridge-lr/test_gen_ridge_lr.parquet', 'test_pred'),
    'heuljax-lr': ('heuljax-lr/oof_logreg.parquet', 'oof_pred', 'heuljax-lr/test_logreg.parquet', 'test_pred'),
    'heuljax-xgb': ('heuljax-xgb/oof_heuljax_xgb.parquet', 'oof_pred', 'heuljax-xgb/test_heuljax_xgb.parquet', 'test_pred'),
    'megayak': ('megayak/oof_hybrid.npy', None, 'megayak/test_hybrid.npy', None),
    'blamerx': ('blamerx/oof.csv', 'pred', 'blamerx/submission.csv', 'Will_Buy_EV'),
}


def load_column(relative_path: str, column: str | None):
    path = OOF_DIR / relative_path
    if column is None:
        values = np.load(path).ravel().astype(float)
    else:
        frame = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
        values = frame[column].values.astype(float)
    return values


oof, test = {}, {}
for name, (oof_file, oof_col, test_file, test_col) in MEMBERS.items():
    oof[name] = load_column(oof_file, oof_col)
    test[name] = load_column(test_file, test_col)

## Hill Climbing

In [ ]:
def hill_climb(ranks: np.ndarray, labels: np.ndarray, steps: int = 150, step_size: float = 0.02):
    """Start beim besten Einzelmitglied; pro Schritt 2% Gewicht zu dem Mitglied schieben, das die AUC am meisten hebt."""
    single = [roc_auc_score(labels, r) for r in ranks]
    weights = np.zeros(len(ranks))
    weights[int(np.argmax(single))] = 1.0
    best = max(single)
    for _ in range(steps):
        best_candidate = None
        for j in range(len(ranks)):
            candidate = weights * (1 - step_size)
            candidate[j] += step_size
            score = roc_auc_score(labels, candidate @ ranks)
            if score > best + 1e-9 and (best_candidate is None or score > best_candidate[0]):
                best_candidate = (score, candidate)
        if best_candidate is None:
            break
        best, weights = best_candidate
    return weights, best


# Rang-Blending: Raenge statt Wahrscheinlichkeiten, damit die unterschiedliche Skala der Modelle keine Rolle spielt
names = list(MEMBERS)
oof_ranks = np.array([rankdata(oof[n]) / len(y) for n in names])
test_ranks = np.array([rankdata(test[n]) / len(test_ids) for n in names])

weights, blend_auc = hill_climb(oof_ranks, y)
print(pd.Series(weights, index=names).sort_values(ascending=False).round(2))  # Final weights
print(f'\nBlend OOF-AUC = {blend_auc:.6f}')

## Submission

In [ ]:
submission = pd.DataFrame({'id': test_ids, TARGET: weights @ test_ranks})
submission.to_csv('submission.csv', index=False)
submission.head()